主要实现的功能：
 - 生成销占比程序表
 - 生成销售预估参数

In [ ]:
import pandas as pd
import os

In [ ]:
# 定义通用的函数

def concate(*args):
    ret = args[0]
    for i in args[1:]:
        ret += ';' + i
    return ret

def replace(x):
    x = str(x)
    x = x.replace('[', '')
    x = x.replace(']', '')
    x = x.replace("'", '')
    return x


def build_and_merge(df, column_name, new_column_name):
    df['temp'] = df['站点'] + '：' + df[column_name]

    locations = df[['站点', '积加SKU', 'temp']]
    locations.drop_duplicates(inplace=True)
    locations = locations.pivot_table(index=['积加SKU'], values='temp', aggfunc=concate)['temp']
    locations = locations.reset_index(level=[0])
    locations['temp'] = locations['temp'].apply(replace)
    locations.rename(columns={'temp': new_column_name}, inplace=True)

    df.drop(columns=['temp'], inplace=True)
    print(locations.head())
    # print(f'columns：{locations.columns}')

    return pd.merge(left=df, right=locations, how='left', left_on=['积加SKU'], right_on=['积加SKU'])

In [ ]:
# 初始化通用变量
path = '../src_data/'

# 1. 销售预估表合并

In [ ]:
sales_predict_files = [f for f in os.listdir(f'{path}销售预估参数') if f.endswith('.xlsx')]

sales_predict = pd.concat(
    [pd.read_excel(f'{path}销售预估参数/{file}', sheet_name='销量预估表-parameter') for file in sales_predict_files],
    # [pd.read_excel(f'{path}销售预估参数/{file}') for file in sales_predict_files],
    ignore_index=True
)
sales_predict.to_excel(f'{path}Listing预估表-模板-CA.xlsx', sheet_name='parms', index=False)

# 只保留US站点的数据
# sales_predict = sales_predict[sales_predict['站点'] == 'US']
sales_predict.to_excel(f'{path}Listing预估表-模板.xlsx', sheet_name='parms', index=False)

In [ ]:
sales_predict.columns

# 2. 销占比参数表处理

## 2.1销占比参数合并

In [ ]:
# sales_files = [f for f in os.listdir(f'{path}销占比参数') if f.endswith('.xlsx')]

# usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
#        'SKU销占比', '需求定位', '发货定位', '备货定位',
#        '店铺', '运营保底下单天数','总发货天数','快递','空运','海运', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间']

# sales = pd.concat(
#     [pd.read_excel(f'{path}销占比参数/{file}', sheet_name='parameter') for file in sales_files],
#     # [pd.read_excel(f'{path}销占比参数/{file}') for file in sales_files],
    
#     ignore_index=True
# )

# sales = sales[usecols]

In [ ]:
sales_files = [f for f in os.listdir(f'{path}销占比参数') if f.endswith('.xlsx')]
sales = pd.concat(
    [pd.read_excel(f'{path}销占比参数/{file}', sheet_name='parameter') for file in sales_files], 
    ignore_index=True
)

In [ ]:
usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
       'SKU销占比', '需求定位', '发货定位', '备货定位',
       '店铺', '运营保底下单天数','总发货天数','快递','空运','海运', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间']
# 输出 快递、空运、海运列为空的行数
# print("快递列为空的行数:", sales['快递'].isnull().sum())
# print("空运列为空的行数:", sales['空运'].isnull().sum())
# print("海运列为空的行数:", sales['海运'].isnull().sum())

# # zhenfei：如果 快递、空运、海运列为空，则 快递 = 6，空运 = 空运在途天数 + 空运打包时间，海运 = 海运在途天数 + 海运打包时间
# sales['快递'] = sales.apply(lambda x: 6 if pd.isnull(x['快递']) else x['快递'], axis=1)
# sales['空运'] = sales.apply(lambda x: x['空运在途天数'] + x['空运打包时间'] if pd.isnull(x['空运']) else x['空运'], axis=1)
# sales['海运'] = sales.apply(lambda x: x['海运在途天数'] + x['海运打包时间'] if pd.isnull(x['海运']) else x['海运'], axis=1)

# # 将海运中含'~'的值用分隔符'~'分割，取第一个值作为海运的值，取第二个值作为 总发货天数1 的值
# sales['总发货天数1'] = sales['海运'].apply(lambda x: str(x).split('~')[1] if '~' in str(x) else x)
# sales['海运'] = sales['海运'].apply(lambda x: str(x).split('~')[0] if '~' in str(x) else x)
# # 如果 总发货天数 的值 小于 总发货天数1 的值，則用 总发货天数1 的值，需要将 总发货天数1 的值转换为数字类型
# sales['总发货天数1'] = sales['总发货天数1'].apply(lambda x: float(x) if pd.notnull(x) else x)
# sales['总发货天数'] = sales.apply(lambda x: x['总发货天数1'] if pd.notnull(x['总发货天数1']) and x['总发货天数1'] < x['总发货天数'] else x['总发货天数'], axis=1)

# # 将空运中含'~'的值用分隔符'~'分割，取第一个值作为空运的值
# sales['空运'] = sales['空运'].apply(lambda x: str(x).split('~')[0] if '~' in str(x) else x)

# # 将快递中含'~'的值用分隔符'~'分割，取第一个值作为快递的值
# sales['快递'] = sales['快递'].apply(lambda x: str(x).split('~')[0] if '~' in str(x) else x)

# # 删除 总发货天数1 列
# sales.drop(columns=['总发货天数1'], inplace=True)
# # 将 快递、空运、海运列转换为数字类型
# sales['快递'] = sales['快递'].astype(float)
# sales['空运'] = sales['空运'].astype(float)
# sales['海运'] = sales['海运'].astype(float)

sales = sales[usecols]
# sales.to_excel(f'{path}Listing销占比-模板-CA-test.xlsx', sheet_name='parms', index=False)

In [ ]:
sales.columns

In [ ]:
# sales.to_excel('C:/Users/gongy/Downloads/销占比/Listing销占比程序表.xlsx')

## 2.2 导入产品信息
[点击查看最新产品信息](https://www.kdocs.cn/l/civjbw2s04Be)

In [ ]:
usecols_for_info = ['MSKU', '店铺/站点', '品牌', 'FNSKU', '仓储类型', '一级品类', '品类A', '品类B']
info = pd.read_excel(f'{path}/产品信息_20231103.xlsx', usecols=usecols_for_info)

In [ ]:
# 唯一性检测，如果存在重复数据则 主动抛出异常
duplicated_cnt = info.duplicated(subset=['MSKU', '店铺/站点']).sum()

if duplicated_cnt > 0:
    duplicated_values = info[info.duplicated(subset=['MSKU', '店铺/站点'], keep=False)][['MSKU', '店铺/站点']]
    raise ValueError(f"检测到产品信息表中存在重复MSKU：\n{duplicated_values}")

info.drop_duplicates(subset=['MSKU', '店铺/站点'], inplace=True)
info.rename(columns={'店铺/站点': '店铺-站点'}, inplace=True)

## 2.2 构建可能用到的维度

In [ ]:
# 拼接 店铺+站点 用于匹配FBA库存 SENWAYZON:CA
sales['店铺-站点'] = sales['店铺'] + ':' + sales['站点']

# 获取 'FNSKU', '仓储类型', '一级品类', '品类A', '品类B'
sales = pd.merge(left=sales, right=info, how='left', on=['店铺-站点', 'MSKU'])

# sales.to_excel('../file/parms_process_sales-原.xlsx', index = False)

In [ ]:
# 使用通用函数构建 定位-集合
sales = build_and_merge(sales, '规模定位','定位-集合')
# sales.to_excel('../file/parms_process_sales-定位-集合.xlsx', index = False)
# 使用通用函数构建 需求定位-集合
sales = build_and_merge(sales, '需求定位','需求定位-集合')
# sales.to_excel('../file/parms_process_sales-需求定位-集合.xlsx', index = False)

In [ ]:
# 输出预览
# sales.query('积加SKU=="RG605 Black L"')
sales.head()

In [ ]:
# 检查根据'MSKU', '店铺/站点'去重，是否存在重复行
duplicated_cnt = sales.duplicated(subset=['MSKU', '店铺-站点']).sum()
print(f'根据MSKU和店铺-站点去重后，重复行数为：{duplicated_cnt}')
if duplicated_cnt > 0:
    duplicated_values = sales[sales.duplicated(subset=['MSKU', '店铺-站点'], keep=False)][['MSKU', '店铺-站点']]
    raise ValueError(f"检测到产品信息表中存在重复MSKU：\n{duplicated_values}")

In [ ]:
# 删除sales结果中站点为“CA”且Listing为“RG802”的行
# sales = sales[~((sales['站点'] == 'CA') & (sales['Listing'] == 'RG802'))]

# 选取站点为US的行
# salas_us = sales[sales['站点'] == 'US']

In [ ]:
# sales.to_excel('../src_data/Listing销占比程序表-模板.xlsx', index=False, sheet_name='parms')
# 读取 月度系数 表
month_df = pd.read_excel(r'E:\sontu\shipping_recommendations\src_data\月度系数\月度系数.xlsx', usecols=['店铺-站点', '月份', 'Listing'])
# 根据 '店铺-站点', 'Listing' 分组，选取最新月份的行
month_df = month_df.sort_values('月份', ascending=False).drop_duplicates(subset=['店铺-站点', 'Listing'], keep='first')

month_sales = pd.merge(left=sales, right=month_df, how='left', on=['店铺-站点', 'Listing'])
# 删除未匹配的行
month_sales = month_sales[month_sales['月份'].notnull()]
# 根据 '店铺-站点', 'Listing' 分组，找出最新的月份值，若该组最新月份值小于当前月份值，则删除整个组的数据
# 当前月份
current_month = pd.Timestamp.today().to_period('M')
# 转换月份，统一为年月格式
month_period = pd.to_datetime(month_sales['月份']).dt.to_period('M')
# 获取每组最新月份
latest_month = month_period.groupby([
    month_sales['店铺-站点'],
    month_sales['Listing']
]).transform('max')
# 保留最新月份 >= 当前月份的整个分组
month_sales = month_sales.loc[latest_month >= current_month].copy()
month_sales.drop(columns=['月份'], inplace=True)
month_sales.to_excel('../src_data/Listing销占比程序表-模板.xlsx', index=False, sheet_name='parms')

In [ ]:
len(month_sales)

In [ ]:
# 销售预估表：根据 '店铺-站点', 'Listing' 分组，选取最新月份的行
sales_predict_month = sales_predict[['站点', 'Listing', '月份']]
sales_predict_month = sales_predict_month.sort_values('月份', ascending=False).drop_duplicates(subset=['站点', 'Listing'], keep='first')
month_predict = pd.merge(left=sales, right=sales_predict_month, how='left', on=['站点', 'Listing'])
print(len(sales))
print(len(month_predict))

# 删除未匹配的行
month_predict = month_predict[month_predict['月份'].notnull()]
print(len(month_predict))

# 根据 '店铺-站点', 'Listing' 分组，找出最新的月份值，若该组最新月份值小于当前月份值，则删除整个组的数据
# 当前月份
current_month = pd.Timestamp.today().to_period('M')
# 转换月份，统一为年月格式
month_period = pd.to_datetime(month_predict['月份']).dt.to_period('M')
# 获取每组最新月份
latest_month = month_period.groupby([
    month_predict['站点'],
    month_predict['Listing']
]).transform('max')
# 保留最新月份 >= 当前月份的整个分组
month_predict = month_predict.loc[latest_month >= current_month].copy()
print(len(month_predict))
# 删除 月份 列
month_predict.drop(columns=['月份'], inplace=True)
month_predict.to_excel('../src_data/Listing销占比程序表-模板-CA.xlsx', index=False, sheet_name='parms')

In [ ]:
sales.columns

In [ ]:
# # 初始化通用变量
# path = 'C:\\Users\\gongy\\Downloads\\'

In [ ]:
# sales_files = [f for f in os.listdir(f'{path}销占比') if f.endswith('.xlsx')]

# # usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
# #        'SKU销占比', '需求定位', '发货定位', '备货定位',
# #        '店铺', '下单天数确认','总发货天数']

# usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
#        'SKU销占比', '需求定位', '发货定位', '备货定位',
#        '店铺', '运营保底下单天数','总发货天数']

# sales = pd.concat(
#     [pd.read_excel(f'{path}销占比/{file}', sheet_name='parameter') for file in sales_files],
#     ignore_index=True
# )


In [ ]:
# sales.to_excel('C:\\Users\\gongy\\Downloads\\销占比/Listing销占比程序表.xlsx', index=False, sheet_name='parms')